# 第 1 章：让资料成为可检索的知识

**本章目标**：选择合适的知识来源与检索单元；比较固定长度、按结构和父子切分；为来源、版本、权限与评估保留元数据。所有代码只读取本地文件。

检索增强生成（RAG, Retrieval-Augmented Generation）最容易被忽视的错误，往往发生在向量计算之前：资料本身不可靠、表格被抽错、关键否定词被切断。假设退款手册写着“仅扣已批准退款；月份按支付时间选订单”，若检索片段只有前半句，模型就无法安全确定时间口径。

## 1.1 先问“什么是知识，什么是事实”

本项目的 <code>db/knowledge.json</code> 保存六张表说明、指标口径、连接规则、时间规则和审核示例。它们回答“如何解释和计算”。订单、支付与退款行留在 PostgreSQL 的 <code>analytics</code> 模式中，由结构化查询语言（SQL, Structured Query Language）回答“此刻具体是多少”。

| 来源 | 合适的处理 | 错误做法 |
|---|---|---|
| 经审核的指标定义 | 作为检索单元，带版本与来源 | 只把指标名嵌入，丢掉状态和时间条件 |
| 当前数据库字段 | 在索引或查询时核对真实模式 | 让模型猜列名 |
| 订单与支付事实行 | 用 SQL 精确过滤和聚合 | 嵌入每笔订单后让模型估算金额 |
| 便携式文档格式（PDF, Portable Document Format）手册 | 解析标题、表格、脚注并保留页码 | 简单去掉换行，破坏表格对应关系 |

生产中还要核对光学字符识别（OCR, Optical Character Recognition）的结果。负号、百分号或“仅”字识别错，后续检索和生成再强也会沿着错误资料推理。

## 1.2 打开项目中的一张指标卡

这里从 JavaScript 对象表示法（JSON, JavaScript Object Notation）文件直接读取，避免连接数据库。项目的 <code>load_knowledge()</code> 在正式索引时还会给表卡追加实时字段清单，所以静态文件不是最终入库文本的全部。

In [1]:
from pathlib import Path
import json
from collections import Counter

root = Path.cwd()
if root.name == "tutorial":
    root = root.parent
root = root.resolve()
cards = json.loads((root / "db" / "knowledge.json").read_text(encoding="utf-8"))
by_id = {card["id"]: card for card in cards}
print("知识卡类型：", dict(Counter(card["kind"] for card in cards)))
metric = by_id["metric.net_sales"]
for key in ("id", "kind", "aliases", "related_tables", "content"):
    print(f"{key}: {metric[key]}")
assert len(by_id) == len(cards)

知识卡类型： {'table': 6, 'metric': 3, 'join': 2, 'rule': 1, 'example': 1}
id: metric.net_sales
kind: metric
aliases: ['净销售额', '净收入', '退款后销售额']
related_tables: ['payments', 'refunds', 'orders']
content: 净销售额：在选定的成功支付订单集合中，SUM(payments.amount) 减去这些订单的 APPROVED 退款金额。先按 order_id 聚合退款再 LEFT JOIN，避免多笔退款放大支付金额。时间窗口默认以 payments.paid_at 选择订单；即使退款发生在窗口之后，也从所选订单金额中扣除。


稳定标识符（ID, Identifier）<code>metric.net_sales</code>用于引用、更新和评估；<code>aliases</code>记录业务同义词；<code>related_tables</code>使检索到指标时能补齐支付、退款、订单表卡。三者有不同职责，不能只保存一大段正文。

本项目把一条完整的业务规则或表定义当成检索单元。对于只有十余张、已人工审核的卡片，这比机械按字符切分更易核对。面对长手册时，才需要更细的片段。

## 1.3 为什么固定长度切分会出错

固定长度加重叠容易实现，适合结构混乱的长文本做第一版基线。但片段可能只保留“已批准退款”，没有“按支付时间选订单”。重叠越大，索引成本与重复证据也越多。下面故意使用很短的窗口制造失败。

In [2]:
rule = "净销售额仅扣除 APPROVED 退款；统计月份按 payments.paid_at 选择成功支付订单。"
width = 20
chunks = [rule[i:i + width] for i in range(0, len(rule), width)]
for number, chunk in enumerate(chunks):
    print(number, repr(chunk))
complete = any("APPROVED" in chunk and "paid_at" in chunk for chunk in chunks)
print("有单个片段同时保留退款状态和时间字段吗？", complete)
print("观察：片段越大越可能保留更多条件，但检索粒度也变粗。")

0 '净销售额仅扣除 APPROVED 退款；'
1 '统计月份按 payments.paid_'
2 'at 选择成功支付订单。'
有单个片段同时保留退款状态和时间字段吗？ False
观察：片段越大越可能保留更多条件，但检索粒度也变粗。


这不是“向量模型不够聪明”，而是索引期已经丢失组合证据。可以把整条规则保存在同一语义单元，或让小片段负责定位、再取回它所属的完整章节。后一种做法称为**父子检索**。

| 切分方案 | 为什么采用 | 优势 | 代价与适用边界 |
|---|---|---|---|
| 固定长度加重叠 | 来源缺少明确结构 | 实现快 | 容易截断条件，重复内容多 |
| 按标题、段落、表格 | 文档结构可靠 | 容易追溯和引用 | 超长章节仍需再细分 |
| 按业务对象成卡 | 指标、表、规则边界清楚 | 语义完整，人工易审核 | 大卡匹配粒度较粗 |
| 父子切分 | 小段好匹配，大段好理解 | 兼顾定位与完整上下文 | 需要父节点映射，扩展后控制词元（token）预算 |
| 问答或摘要增强 | 用户问法与原文差异大 | 扩展可命中的表达 | 自动生成摘要可能写错关键规则 |

方案好坏要在代表性问题上比较“必要证据是否被召回”，不能只看切出的片段平均长度。

## 1.4 一个最小父子检索实验

下面把“退款手册第 3 节”看作父节点，两个子片段负责匹配。命中其中一句时，返回完整章节。项目当前的 <code>related_tables</code> 是依赖补全，思路相近，但还没有真正的父子片段表。

In [3]:
parents = {
    "refunds.section3": "退款手册第 3 节：净销售额仅扣除 APPROVED 退款；统计月份按 payments.paid_at 选择订单。"
}
children = [
    {"chunk_id": "refunds.3.1", "parent_id": "refunds.section3",
     "text": "净销售额仅扣除 APPROVED 退款"},
    {"chunk_id": "refunds.3.2", "parent_id": "refunds.section3",
     "text": "统计月份按 payments.paid_at 选择订单"},
]
hits = [child for child in children if "APPROVED" in child["text"]]
returned_context = [parents[hit["parent_id"]] for hit in hits]
print("命中子片段：", [hit["chunk_id"] for hit in hits])
print("返回父章节：", returned_context)
assert "paid_at" in returned_context[0]

命中子片段： ['refunds.3.1']
返回父章节： ['退款手册第 3 节：净销售额仅扣除 APPROVED 退款；统计月份按 payments.paid_at 选择订单。']


## 1.5 元数据决定知识能否安全使用

一条可运营的索引记录通常有 <code>source_id</code>、<code>chunk_id</code>、<code>parent_id</code>、内容哈希、来源版本、生效日期与访问范围。统一资源定位符（URL, Uniform Resource Locator）用于记录来源地址，把答案指回原文。访问控制列表（ACL, Access Control List）说明哪些用户能读到哪些资料。

**权限过滤要发生在召回之前或召回过程中。** 如果先从所有租户的文档中挑前 10 条，再删除无权查看的结果，本租户可能没有足够候选；若先把无权内容交给重排序模型，已经泄露。当前项目的 13 张教学卡没有租户 ACL，下面只是离线设计实验，不代表生产代码已有该能力。

In [4]:
documents = [
    {"id": "a", "tenant": "east", "text": "华东退款规则"},
    {"id": "b", "tenant": "south", "text": "华南私有口径"},
    {"id": "c", "tenant": "east", "text": "华东支付时间规则"},
]
request_tenant = "east"
visible = [item for item in documents if item["tenant"] == request_tenant]
print("可进入检索的文档：", [item["id"] for item in visible])
assert {item["id"] for item in visible} == {"a", "c"}

可进入检索的文档： ['a', 'c']


## 练习

**基础练习**：从 <code>db/knowledge.json</code> 找出“净销售额”题至少需要的指标卡与表卡；解释为什么不能把每笔订单作为知识卡。再把第 1.3 节的窗口从 20 改成 40，观察关键条件是否始终落在同一片段。**检查标准**：指出 <code>metric.net_sales</code> 与支付、退款表；解释窗口变大可能减少断裂，却不保证所有文档都安全。

**进阶练习**：为一份 20 页退款手册设计 <code>source_id / parent_id / chunk_id</code>，写出“子片段命中后取回父章节”的规则，并限定最多取回多少词元。再设计两条问题检验脚注和表格是否被正确取回。**检查标准**：答案能追溯到原页；至少包含一条否定或状态条件；权限过滤发生在候选生成前。

**自检问题**：如果原始 PDF 把金额 <code>-100</code> 识别成 <code>100</code>，增加重排序能修复吗？不能。首先应核对抽取结果与原始页。

## 继续阅读

- [Microsoft 的 RAG 分块阶段指南](https://learn.microsoft.com/en-us/azure/architecture/ai-ml/guide/rag/rag-chunking-phase)
- [项目知识准备代码](../src/nl2sql/knowledge.py)